In [14]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from bs4 import BeautifulSoup

# Jupyter cwd is this notebook folder, not the repo root.
# week1/ is a directory with scraper.py, not an importable package named week1.
# day1.ipynb is a notebook, so messages_for cannot be imported from it.
here = Path.cwd().resolve()
week1_dir = None
for path in [here, *here.parents]:
    if (path / "scraper.py").is_file() and path.name == "week1":
        week1_dir = path
        break
    if (path / "week1" / "scraper.py").is_file():
        week1_dir = path / "week1"
        break
if week1_dir is None:
    raise FileNotFoundError("Could not find week1/scraper.py from this notebook")
sys.path.insert(0, str(week1_dir))

from scraper import fetch_website_contents

load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
api_key = os.getenv("GOOGLE_API_KEY")

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

system_prompt = (
    "You are a Thug assistant that analyzes the contents of a website, and provides a summary "
    "like Ice Cube's character from the Superbad movie. Analyses should be as funny as his "
    "character, ignoring text that might be navigation related. Respond in markdown. Do not wrap "
    "the markdown in a code block - respond just with the markdown."
)
user_prompt_prefix = """
Please analyze the following website:

"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website},
    ]

ed = fetch_website_contents("https://edwarddonner.com")
messages = messages_for(ed)

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)
print(ed)
display(Markdown(response.choices[0].message.content))


ModuleNotFoundError: No module named 'week1.day1'